# 使用 OpenAI Agents SDK 进行多智能体编排：金融投资组合分析示例

## 简介

*本指南面向已经熟悉 OpenAI 模型和 LLM 智能体、并希望了解如何编排一组智能体来完成真实复杂任务的读者。*

**你将学到什么**

在本 notebook 中，你将学习如何使用 OpenAI Agents SDK 设计并实现一个复杂的多智能体协作系统。具体来说，你将了解如何：
- 构建一个工作流，让多个专家智能体（宏观、基本面、量化）在投资组合经理智能体的统筹下协作，解决一个具有挑战性的投资研究问题。
- 使用“智能体即工具”（agents as a tool）方法，由一个中心智能体进行编排，并将其他智能体作为工具调用来完成特定子任务。
- 在一个集成的工作流中，利用 SDK 支持的所有主要工具类型（自定义 Python 函数、Code Interpreter 和 WebSearch 等托管工具，以及外部 MCP 服务器）。
- 在智能体模式中应用模块化、并行化和可观测性方面的最佳实践。

**为什么这很重要**

“智能体即工具”模式是构建透明、可审计、可扩展的多智能体协作的一种强大方式。本示例演示了如何使用 OpenAI Agents SDK 将深度专业分工、并行执行和稳健的编排结合起来。

读完本指南后，你将获得一份清晰的蓝图，用于构建自己的多智能体工作流，适用于研究、分析或任何能从专家协作中受益的复杂任务。



---

## 目录

1. [什么是多智能体协作？](#什么是多智能体协作)
2. [协作模式：Handoff 与 Agent-as-Tool](#协作模式handoff-与-agent-as-tool)
3. [架构概览](#架构概览)
4. [支持的工具类型](#支持的工具类型)
5. [设置](#设置)
6. [运行工作流](#运行工作流)
7. [首席投资组合经理（PM）智能体](#the-head-portfolio-manager-pm-agent)
8. [拆解首席投资组合经理智能体](#拆解首席投资组合经理智能体)
9. [示例输出](#示例输出)
10. [构建智能体的最佳实践](#构建智能体的最佳实践)
11. [延伸阅读与最佳实践](#延伸阅读与最佳实践)



---

## 什么是多智能体协作？

**多智能体协作**是指多个自主智能体（LLM“节点”）相互协调，以实现单个智能体难以完成的总体目标。与使用一个庞大的单体提示词不同，每个智能体负责一个特定的子任务或专业领域，而编排层将这些智能体“节点”连接成一个连贯的工作流。这种方法对复杂系统很有用。例如，一项金融分析可以拆分为宏观经济分析、公司基本面分析和量化信号分析，每一项都由不同的专家智能体处理。各智能体共享信息，并将各自的结果组合起来，得出最终结果。



### 协作模式：Handoff 与 Agent-as-Tool

OpenAI Agents SDK 支持多种智能体协作模式：

- **交接（Handoff）协作：** 一个智能体可以在处理问题的过程中将控制权 _交接_ 给另一个智能体。在交接架构中，每个智能体都知道其他智能体的存在，并可以决定何时转交给更合适的智能体。这对于开放式或对话式工作流很灵活，但可能更难保持对任务的全局视图。[在 SDK 文档中了解更多。](https://openai.github.io/openai-agents-python/handoffs/)

- **智能体即工具（Agent as a Tool）：** 在这种方法中，一个智能体（通常是中心规划者或管理者）**像调用工具一样调用其他智能体**。子智能体不会接管对话；而是由主智能体针对特定子任务调用它们，并整合其结果。这种模式保持单一的控制线程（由主智能体编排一切），往往能简化协调工作。**本仓库采用智能体即工具模式：** 投资组合经理智能体始终处于主导地位，在需要专业知识时将其他专家智能体作为工具使用。这一选择让整体推理保持透明，并允许子任务并行执行，非常适合复杂分析。

有关这些协作模式的更多信息，请参阅 [OpenAI Agents SDK 文档](https://openai.github.io/openai-agents-python/multi_agent/)。

---

## 架构概览

我们的系统采用**中心辐射式设计**（hub-and-spoke）。**投资组合经理智能体**是中心（中央协调者），**专家智能体**是辐条。用户的查询（例如“计划中的降息会如何影响我持有的 GOOGL？”）首先发送给投资组合经理。投资组合经理智能体会被提示拆解问题，并委派给合适的专家智能体。它将每个专家视为可调用的工具，调用它们完成各自负责的分析部分。三个专家都向投资组合经理汇报，然后由投资组合经理为用户综合出最终答案。

![多智能体投资报告工作流](../../../images/multi_agent_collab_agent_architecture.png)



---

## 支持的工具类型

Agents SDK 的一个关键优势是可以灵活定义智能体能够使用的**工具**。工具既可以是简单的 Python 函数，也可以是外部服务。在本项目中，我们使用：

- **MCP（Model Context Protocol）服务器：** 用于以标准化的方式将智能体连接到外部工具和数据源。本项目使用一个本地 MCP 服务器获取 Yahoo Finance 数据（参见 `mcp/yahoo_finance_server.py`）。[了解更多：OpenAI MCP 文档](https://openai.github.io/openai-agents-python/mcp/) | [MCP 规范](https://modelcontextprotocol.io/)

- **OpenAI 托管工具：** 托管工具是 OpenAI 提供的内置托管工具，无需自定义实现。它们开箱即提供强大的能力，例如 **Code Interpreter**（用于量化/统计分析）和 **WebSearch**（用于获取最新新闻和数据）。这些工具易于集成，由 OpenAI 维护，让智能体无需额外设置即可执行代码运行和实时信息检索等高级操作。

- **自定义工具：** 自定义工具是你定义并注册为智能体工具的任意 Python 函数。Agents SDK 让这件事变得很简单：只需给函数加上装饰器，SDK 就会自动提取其名称、docstring 和输入 schema。这非常适合领域特定逻辑、数据访问或工作流扩展。 
  在我们的项目中，我们使用自定义工具访问 FRED 经济数据（[参见 FRED API](https://fred.stlouisfed.org/docs/api/api_key.html)）并执行文件系统操作。

自定义工具让你可以完全灵活地将智能体的能力扩展到内置或托管工具之外。[参阅 SDK 中关于函数工具的文档。](https://openai.github.io/openai-agents-python/tools/#function-tools)

> **想添加更多工具？** SDK 支持多种工具类型，包括网页搜索、文件搜索、代码执行等。[在 SDK 文档中查看受支持工具的完整列表。](https://openai.github.io/openai-agents-python/tools/)

---

## 设置

In [ ]:
# Install required dependencies
!pip install -r requirements.txt

**运行工作流之前，请设置以下环境变量：**
- `OPENAI_API_KEY`（用于访问 OpenAI）
- `FRED_API_KEY`（用于获取 FRED 经济数据，参见 [FRED API 密钥说明](https://fred.stlouisfed.org/docs/api/api_key.html)）

In [ ]:
import os

missing = []
if not os.environ.get('OPENAI_API_KEY'):
    missing.append('OPENAI_API_KEY')
if not os.environ.get('FRED_API_KEY'):
    missing.append('FRED_API_KEY')

if missing:
    print(f"Missing environment variable(s): {', '.join(missing)}. Please set them before running the workflow.")
else:
    print("All required API keys are set.")

---

## 运行工作流 

你可以随意修改问题，但请保留日期字段以提高准确性！

<div style="border-left: 4px solidrgb(0, 0, 0); padding: 0.5em; background:rgb(255, 229, 229);">
<strong>免责声明：</strong>本示例仅用于教学目的。在做出任何投资决策之前，请咨询合格的金融专业人士
</div>


工作流通过向首席投资组合经理（PM）智能体发送用户请求来启动。PM 智能体编排整个流程，并根据需要委派给专家智能体和工具。你可以使用 OpenAI Traces 实时监控工作流，它能详细展示每一次智能体和工具调用。

你可以随意修改下方代码中的 `question`，但请保留日期字段以提高准确性！

<div style="border-left: 4px solid #f39c12; padding: 0.5em; background: #fffbe6;">
<strong>注意：</strong>根据任务的复杂程度，此请求最多可能需要 10 分钟。
</div>


In [ ]:
import datetime
import json
import os
from pathlib import Path
from contextlib import AsyncExitStack
from agents import Runner, add_trace_processor, trace
from agents.tracing.processors import BatchTraceProcessor
from utils import FileSpanExporter, output_file
from investment_agents.config import build_investment_agents
import asyncio

add_trace_processor(BatchTraceProcessor(FileSpanExporter()))

async def run_workflow():
    if "OPENAI_API_KEY" not in os.environ:
        raise EnvironmentError("OPENAI_API_KEY not set — set it as an environment variable before running.")

    today_str = datetime.date.today().strftime("%B %d, %Y")
    question = (
        f"Today is {today_str}. "
        "How would the planned interest rate reduction effect my holdings in GOOGL if they were to happen?"
        "Considering all the factors effecting its price right now (Macro, Technical, Fundamental, etc.), what is a realistic price target by the end of the year?"
    )
    bundle = build_investment_agents()

    async with AsyncExitStack() as stack:
        for agent in [getattr(bundle, "fundamental", None), getattr(bundle, "quant", None)]:
            if agent is None:
                continue
            for server in getattr(agent, "mcp_servers", []):
                await server.connect()
                await stack.enter_async_context(server)

        print("Running multi-agent workflow with tracing enabled...\n")
        with trace(
            "Investment Research Workflow",
            metadata={"question": question[:512]}
        ) as workflow_trace:
            print(
                f"\n🔗 View the trace in the OpenAI console: "
                f"https://platform.openai.com/traces/trace?trace_id={workflow_trace.trace_id}\n"
            )

            response = None
            try:
                response = await asyncio.wait_for(
                    Runner.run(bundle.head_pm, question, max_turns=40),
                    timeout=1200
                )
            except asyncio.TimeoutError:
                print("\n❌ Workflow timed out after 20 minutes.")

            report_path = None
            try:
                if hasattr(response, 'final_output'):
                    output = response.final_output
                    if isinstance(output, str):
                        data = json.loads(output)
                        if isinstance(data, dict) and 'file' in data:
                            report_path = output_file(data['file'])
            except Exception as e:
                print(f"Could not parse investment report path: {e}")

            print(f"Workflow Completed Response from Agent: {response.final_output if hasattr(response, 'final_output') else response}, investment report created: {report_path if report_path else '[unknown]'}")

# In a Jupyter notebook cell, run:
await run_workflow()


---

## 拆解首席投资组合经理智能体

首席投资组合经理（PM）智能体是整个工作流的编排者。它协调一组共四个专家智能体，每个智能体专注于不同的专业领域。这种设计是有意为之的：让单个智能体承担所有可能的职责，会导致输出肤浅、泛泛，也会让系统随着时间推移难以维护或改进。

### 为什么采用这种设计？
通过将问题拆分给职责明确的专家智能体，你可以获得：

- **更深入、更高质量的研究：** 每个智能体都可以专注于自己的领域，使用适合该工作的工具和提示词。PM 智能体将这些视角汇集起来，得出更细致、更稳健的答案。

- **模块化与清晰性：** 你可以更新、测试或改进某一个智能体，而不影响其他智能体。这让系统随着需求变化更易于维护和扩展。

- **并行带来更快的结果：** 相互独立的智能体可以同时工作，大幅缩短完成复杂、多部分分析所需的时间。

- **一致性与可审计性：** 结构化的、由提示词驱动的工作流确保每次运行都遵循最佳实践、易于调试，并产出值得信赖、可供审阅的结果。

对于任何需要深度、专业分工和可靠性的应用，这种方法都是理想之选，无论你是在构建研究助手、决策支持工具，还是任何能从专家协作与编排中受益的系统。

**我们在实践中如何实现：**
- 每个专家智能体（基本面、宏观、量化）都使用 SDK 的 `function_tool` 装饰器封装为可调用工具，并带有自定义名称和描述。这让 PM 智能体的工具集清晰明确，也便于 LLM 理解。

- 首席 PM 智能体使用 `run_all_specialists_parallel` 工具并发调用全部三个专家，并利用 `parallel_tool_calls=True` 实现最高的速度和效率。

- 智能体的提示词从一个 markdown 文件（`pm_base.md`）加载，其中不仅编码了公司的投资理念，还包含详细的工具使用规则和分步工作流。这确保每次运行都一致、可审计，并符合最佳实践。

- 在收集并审阅专家的输出后，PM 智能体使用一个专门的备忘录编辑工具来汇总、排版并最终完成投资报告。这种关注点分离让工作流保持模块化，易于扩展。

- 系统在设计上便于扩展：你可以添加新的专家智能体、替换工具或更新提示词，而不会破坏整体编排逻辑。所有工具调用、智能体决策和输出都会记录在 OpenAI Traces 中，实现完全透明和便于调试。

这些实现选择直接支撑了上述优势，使深入、模块化且可靠的多智能体研究工作流易于维护、审计和改进。

### 首席投资组合经理智能体：代码

```python
from agents import Agent, ModelSettings, function_tool
from utils import load_prompt, DISCLAIMER

def build_head_pm_agent(fundamental, macro, quant, memo_edit_tool):
    def make_agent_tool(agent, name, description):
        @function_tool(name_override=name, description_override=description)
        async def agent_tool(input):
            return await specialist_analysis_func(agent, input)
        return agent_tool
    fundamental_tool = make_agent_tool(fundamental, "fundamental_analysis", "Generate the Fundamental Analysis section.")
    macro_tool = make_agent_tool(macro, "macro_analysis", "Generate the Macro Environment section.")
    quant_tool = make_agent_tool(quant, "quantitative_analysis", "Generate the Quantitative Analysis section.")

    @function_tool(name_override="run_all_specialists_parallel", description_override="Run all three specialist analyses (fundamental, macro, quant) in parallel and return their results as a dict.")
    async def run_all_specialists_tool(fundamental_input, macro_input, quant_input):
        return await run_all_specialists_parallel(
            fundamental, macro, quant,
            fundamental_input, macro_input, quant_input
        )

    return Agent(
        name="Head Portfolio Manager Agent",
        instructions=(load_prompt("pm_base.md") + DISCLAIMER),
        model="gpt-4.1",
        tools=[fundamental_tool, macro_tool, quant_tool, memo_edit_tool, run_all_specialists_tool],
        model_settings=ModelSettings(parallel_tool_calls=True, tool_choice="auto", temperature=0)
    )
    ```

### 首席 PM 系统提示词：落实最佳实践

PM 智能体的系统提示词（参见 `prompts/pm_base.md`）是整个工作流的核心。它编码了：
- 公司的投资理念（原创性、风险意识、挑战市场共识）
- 清晰的工具使用规则（何时使用并行工具、如何组织输入）
- 稳健的多步骤工作流（确定任务类型、提供指导、审阅输出、汇编备忘录、处理缺失数据）

该提示词确保每次运行都：
- **一致：** 每次都遵循同样的高标准和流程。
- **可审计：** 每个步骤、工具调用和决策都在追踪记录中可见。
- **高质量：** 输出具有原创性、风险意识，并经过严格审阅。

In [ ]:
# Render the actual system prompt used by the Head Portfolio Manager agent
from pathlib import Path
from IPython.display import Markdown, display

pm_prompt_path = Path("prompts/pm_base.md")
if pm_prompt_path.exists():
    with pm_prompt_path.open("r", encoding="utf-8") as f:
        content = f.read()
    display(Markdown(content))
else:
    print("System prompt not found at prompts/pm_base.md")

---

## 示例输出

下面是通过该工作流生成的一份投资报告示例。你的输出将写入目录中的 `outputs` 文件夹。 

<details>
<summary>点击展开投资备忘录</summary>

# 投资备忘录：Alphabet Inc.（GOOGL）——计划降息的影响（2025 年 5 月）

## 执行摘要

Alphabet Inc.（GOOGL）当前股价为每股 \$171.42，市值 \$1.88 万亿，市盈率 16.91。投资论点为温和建设性：虽然美联储计划中的降息是一个温和的利好，但它并不是 GOOGL 股价走势的主要驱动因素。最具原创性、差异化的洞见（与本公司愿景完全一致）是：GOOGL 对利率的直接敏感度不高（与 10 年期国债收益率的最大周相关性约为 0.29），真正的风险/回报取决于 AI 驱动增长的可持续性、板块轮动以及监管阻力。这一论点得到了稳健的技术面、强劲的基本面和压倒性正面的分析师情绪的支持，但也受到 AI 乐观情绪消退或宏观/监管冲击出现的风险制约。共识观点有证据支撑：GOOGL 的业务依然具有韧性，但不同观点（即降息未能提振科技股，或板块轮动限制回报）也不应被忽视。主要风险包括监管行动、宏观经济不确定性以及 AI 叙事可能发生转变。在最好情况下，GOOGL 到 2025 年底可能达到 \$200–\$210；在最坏情况下，回测 \$160–\$170 也是有可能的。本备忘录通过聚焦情景规划、原创量化分析以及对共识与不同观点的批判性评估，体现了公司的愿景。

## 基本面视角

Alphabet 的核心业务由其在数字广告领域（Google Search、YouTube）的主导地位以及不断增长的云和 AI 业务驱动。截至最近一个季度（2025 年第一季度），营收为 \$902 亿，净利润 \$345 亿，每股收益 \$2.81，净利率 38.3%。过去一年利润率有所改善，公司的规模以及在 AI 和云领域的领先地位构成了持久的护城河。然而，近期分析师目标价被下调（Bernstein：\$165，UBS：\$209，Wolfe：\$210），反映出对监管和宏观经济风险的谨慎态度。共识观点是合理的：虽然 Alphabet 的财务实力和创新能力显而易见，但监管审查和宏观逆风（例如经济下行时广告预算缩减）都是真实存在的风险。最具原创性的洞见是公司的适应和创新能力，这可能会缓解部分风险。该分析以证据为基础，近期季度数据显示利润率保持稳定或有所改善：

| 日期       |    营收 |   净利润 |   毛利 |   总费用 |   EPS |   净利率 (%) |   毛利率 (%) |   营业利润率 (%) |
|:-----------|-----------:|-------------:|---------------:|-----------------:|------:|-----------------:|-------------------:|-----------------------:|
| 2025-03-31 | 9.0234e+10 |   3.454e+10  |     5.3873e+10 |       5.9628e+10 |  2.81 |          38.28 |            59.70 |                33.92 |
| 2024-12-31 | 9.6469e+10 |   2.6536e+10 |     5.5856e+10 |       6.5497e+10 |  2.15 |          27.51 |            57.90 |                32.11 |
| 2024-09-30 | 8.8268e+10 |   2.6301e+10 |     5.1794e+10 |       5.9747e+10 |  2.12 |          29.80 |            58.68 |                32.31 |
| 2024-06-30 | 8.4742e+10 |   2.3619e+10 |     4.9235e+10 |       5.7317e+10 |  1.89 |          27.87 |            58.10 |                32.36 |
| 2024-03-31 | 8.0539e+10 |   2.3662e+10 |     4.6827e+10 |       5.5067e+10 |  1.89 |          29.38 |            58.14 |                31.63 |

近期分析师情绪压倒性地正面，目前有 56 个买入、12 个持有和 0 个卖出评级：

| 时期       |   买入 |   持有 |   卖出 |
|:-------------|------:|-------:|-------:|
| 当前      |    56 |     12 |      0 |
| 1 个月前  |    55 |     12 |      0 |
| 2 个月前 |    55 |     12 |      0 |
| 3 个月前 |    53 |     12 |      0 |

基本面观点通过聚焦证据、情景规划而非简单跟随共识，与公司愿景保持一致。与公司愿景的主要偏差可能在于分析未能考虑监管或宏观冲击的影响，但本文已对此进行了讨论。

## 宏观视角

宏观经济环境好坏参半。美国实际 GDP 正在扩张（\$23.5 万亿，2025 年第一季度），失业率较低（4.2%），通胀仍处于高位（CPI：320.3）。美联储将利率维持在 4.25–4.50%，态度保持耐心，并关注不断演变的风险。美元走强（DXY：123.4），近期的关税措施带来了不确定性。投资者正从美国科技股转向亚洲股票，反映出对高估值的担忧以及对海外更好增长前景的预期。宏观层面的共识观点是降息将支撑科技股估值，但不同观点（得到本公司愿景的支持）认为，板块轮动和贸易政策可能会抵消这些好处。尾部风险情景包括：基准情景下降息支撑 GOOGL（目标价 \$180–\$190），下行情景下贸易紧张或板块轮动限制回报。该分析以证据为基础，使用了 FRED 数据和近期的政策声明，并明确考虑了最好和最坏两种情景。宏观观点通过挑战共识并为多种结果做规划，与公司愿景完全一致。

## 量化视角

量化分析证实，GOOGL 对利率的直接敏感度不高。其与 10 年期美国国债收益率的平均周相关性为 0.29，与联邦基金利率的平均周相关性为 0.05，表明利率变化并不是 GOOGL 回报的主要驱动因素。技术面稳健：GOOGL 位于关键移动平均线上方，动量为正，波动率适中。情景分析显示，降息是一个温和的利好，但如果这一变动已被市场定价，或者技术面破位，则可能出现 5–10% 的回调。分析师情绪强烈正面，基本面（营收、利润率）正在改善。量化汇总统计如下：

| 指标                                  |     数值 |
|:----------------------------------------|----------:|
| 平均日相关性 (FEDFUNDS, GOOGL)       | 0.05 |
| 平均日回归斜率 (FEDFUNDS, GOOGL)  | 0.02 |
| 平均日相关性 (DGS10, GOOGL)          | 0.13 |
| 平均日回归斜率 (DGS10, GOOGL)     | 0.05 |
| 平均周相关性 (FEDFUNDS, GOOGL)      | 0.05 |
| 平均周回归斜率 (FEDFUNDS, GOOGL) | 0.03 |
| 平均周相关性 (DGS10, GOOGL)         | 0.29 |
| 平均周回归斜率 (DGS10, GOOGL)    | 0.09 |

关键图表和图像：

![GOOGL 日收益率](../../../images/multi_agent_collab_googl_daily_returns.png)
![GOOGL 移动平均线](../../../images/multi_agent_collab_googl_moving_averages.png)
![GOOGL RSI](../../../images/multi_agent_collab_googl_rsi.png)
![GOOGL 滚动波动率](../../../images/multi_agent_collab_googl_rolling_volatility.png)
![累计收益率对比](../../../images/multi_agent_collab_cumulative_return_comparison.png)
![滚动波动率对比](../../../images/multi_agent_collab_rolling_volatility_comparison.png)
![滚动相关性/回归（日度，联邦基金利率）](../../../images/multi_agent_collab_rolling_corr_reg_daily_fedfunds.png)
![滚动相关性/回归（日度，10 年期）](../../../images/multi_agent_collab_rolling_corr_reg_daily_dgs10.png)
![滚动相关性/回归（周度，联邦基金利率）](../../../images/multi_agent_collab_rolling_corr_reg_weekly_fedfunds.png)
![滚动相关性/回归（周度，10 年期）](../../../images/multi_agent_collab_rolling_corr_reg_weekly_dgs10.png)
![GOOGL 季度趋势](../../../images/multi_agent_collab_GOOGL_quarterly_trends.png)
![GOOGL 季度利润率](../../../images/multi_agent_collab_GOOGL_quarterly_margins.png)
![GOOGL 分析师评级趋势](../../../images/multi_agent_collab_GOOGL_analyst_recommendations_trend.png)

量化观点的原创性在于聚焦情景分析和不高的利率敏感度，并且通过不简单跟随共识而与公司愿景保持一致。其局限性包括疫情后的数据窗口较短，以及 GOOGL 的股价受利率之外的多种因素（AI、广告市场、监管）驱动。

## 投资组合经理视角

PM 的综合结论是：三个专家部分都指向温和建设性的前景，2025 年底的现实目标价为 \$190–\$210。最具原创性的洞见是 GOOGL 对利率的直接敏感度不高，真正的风险在于 AI 驱动的增长能否持续，或者板块轮动和监管阻力是否会限制回报。量化部分有力地指出了稳健的技术面和市场情绪，但也提示了在下行情景中回测 \$160–\$170 的风险。基本面和宏观部分强调了监测监管和贸易政策的重要性。如果目前低配大盘科技股，现在是一个合理的入场时机，但仓位规模应反映板块轮动或宏观表现不及预期的风险。不同观点（即降息未能提振科技股，或 AI 叙事发生转变）不应被忽视。仓位规模和风险管理是关键，这与公司注重情景规划和差异化洞见的愿景完全一致。

## 建议与问题解答

建议维持或适度增加对 GOOGL 的敞口，尤其是在低配大盘科技股的情况下，基准情景下 2025 年底的目标价为 \$200–\$210。这通过聚焦原创的、以证据为基础的情景分析，而非简单跟随共识，体现了公司的愿景。该建议有稳健的基本面、积极的技术面和强烈的分析师情绪作为依据，但也受到板块轮动、监管行动或 AI 叙事转变等风险的制约。如果这些风险成为现实，可能会回测 \$160–\$170。仓位规模和风险管理应反映这些情景。这种方法具有差异化、以证据为驱动，并与公司愿景完全一致。

**END_OF_MEMO**

*免责声明：我是一个 AI 语言模型，不是注册投资顾问。所提供的信息仅具有教育性质和一般性质。在做出任何投资决策之前，请咨询合格的金融专业人士。*

</details>


## 构建智能体的最佳实践

最有效的智能体系统结合了模块化的智能体设计、清晰的工具定义、并行执行和结构化的提示词。这种方法是 OpenAI Agents SDK 的核心，它让你的工作流稳健、可扩展，并且易于调试或扩展。

**OpenAI Agents SDK 中支撑这些最佳实践的关键特性：**
- **智能体循环：** 自动处理工具调用、LLM 推理和工作流控制。
- **以 Python 为先的编排：** 使用熟悉的 Python 模式来串联、组合和编排智能体。
- **交接（Handoffs）：** 在智能体之间委派任务，实现专业分工和模块化。
- **护栏（Guardrails）：** 验证输入/输出，并在出错时尽早中断，以提高可靠性。
- **函数工具：** 将任意 Python 函数注册为工具，并自动生成 schema 和进行验证。
- **追踪（Tracing）：** 可视化、调试和监控工作流的每一步，实现完全透明。

精心设计的工具、周密的编排和审慎的模型选择相结合，对于构建有效的智能体系统至关重要。在本示例中，我们使用 GPT-4.1 系列模型，因为它们具备强大的分析和工具使用能力（[参见 GPT-4.1 提示词指南](https://cookbook.openai.com/examples/gpt4-1_prompting_guide)）。如需了解更深入的架构最佳实践，请参阅随附的[《构建智能体实用指南》（PDF）](https://cdn.openai.com/business-guides-and-resources/a-practical-guide-to-building-agents.pdf)。将这些要素结合在一起，你就能得到一个稳健、可扩展、易于调试或扩展的系统。

欢迎用你自己的投资问题试用这个示例，也欢迎分享任何反馈！祝构建愉快。

---

## 延伸阅读与最佳实践

- [OpenAI Agents SDK 文档](https://openai.github.io/openai-agents-python/)
- [OpenAI Agents SDK：多智能体编排](https://openai.github.io/openai-agents-python/multi_agent/)
- [OpenAI Agents SDK：工具列表](https://openai.github.io/openai-agents-python/tools/)
- [OpenAI Agents SDK：MCP 文档](https://openai.github.io/openai-agents-python/mcp/)

- [MCP 规范](https://spec.modelcontextprotocol.io/specification/2024-11-05/architecture/)
- [OpenAI Cookbook](https://github.com/openai/openai-cookbook)
- （[GPT-4.1 提示词指南](https://cookbook.openai.com/examples/gpt4-1_prompting_guide)）
- [《构建智能体实用指南》（PDF）](https://cdn.openai.com/business-guides-and-resources/a-practical-guide-to-building-agents.pdf)

---